# Plazas de parking libres y detección de objetos

Notebook ejecutable del miniproyecto. Orquesta la preparación de PKLot, los baselines de ocupación, YOLOv8n, DINOv2, la fusión y la evaluación. Reutiliza el paquete `parking_project` que acompaña a este notebook.

Regla: un coche o una moto asociados a una plaza implican ocupación; las personas se detectan y evalúan aparte.

## 1. Entorno

Instala primero las dependencias base del curso y sigue `INSTALL.md` para PyTorch/torchvision y Ultralytics. La primera inferencia descarga los pesos si todavía no están en caché. Para la RTX 2070 Super, selecciona en la página oficial de PyTorch la variante CUDA compatible con el controlador instalado.

In [1]:
from pathlib import Path
import csv, json, shutil, sys
import cv2
import matplotlib.pyplot as plt

# Encuentra el proyecto si Jupyter se inició aquí, en Prácticas/ o en la raíz.
cwd = Path.cwd()
candidates = [cwd, *cwd.parents, cwd / 'Prácticas' / 'ProyectoParking', cwd / 'ProyectoParking']
PROJECT_DIR = next((p.resolve() for p in candidates if (p / 'parking_project').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('No encuentro ProyectoParking. Abre el notebook desde el proyecto o ajusta PROJECT_DIR.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from parking_project.dataset import prepare_dataset, read_manifest
from parking_project.pipeline import run_experiment

CONFIG_PATH = PROJECT_DIR / 'config.json'
if not CONFIG_PATH.exists():
    shutil.copyfile(PROJECT_DIR / 'config.example.json', CONFIG_PATH)
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
OUTPUT_DIR = PROJECT_DIR / 'outputs'
def resolve_project_path(value):
    path = Path(value).expanduser()
    return str((path if path.is_absolute() else PROJECT_DIR / path).resolve())
print('Proyecto:', PROJECT_DIR)
print('Configuración:', CONFIG_PATH)

Proyecto: C:\Users\Roland\Desktop\Universidad Master IA\VA - Visión artifical\ProyectoParking
Configuración: C:\Users\Roland\Desktop\Universidad Master IA\VA - Visión artifical\ProyectoParking\config.json


## 2. Preparar la muestra

Descarga y extrae PKLot fuera del repositorio. `DATASET_ROOT` debe ser la carpeta que contiene `parking1a`, `parking1b` y `parking2` (o `UFPR04`, `UFPR05` y `PUCPR`). Dentro hay subcarpetas de clima y fecha, con cada imagen de escena completa junto a su XML. No selecciones `PKLotSegmented`, que contiene recortes de plazas. Por ejemplo, si al extraer queda `D:/Datos/PKLot/PKLot/parking1a`, asigna `DATASET_ROOT = Path('D:/Datos/PKLot/PKLot')`.

La página oficial ofrece 12.417 imágenes con anotaciones XML. Este notebook selecciona hasta 600 imágenes de desarrollo y 150 de `parking2`/`PUCPR` como vista externa, equilibrando vista, clima y día; los días quedan completos dentro de un único split.

In [2]:
# Edita la ruta antes de ejecutar. Ejemplo Windows: Path(r'D:\\datasets\\PKLot')
DATASET_ROOT = Path(r'RUTA/AL/PKLot/EXTRAIDO')
PREPARED_DIR = OUTPUT_DIR / 'prepared'
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Edita DATASET_ROOT para apuntar a PKLot: {DATASET_ROOT}')
MANIFEST_PATH, PREPARATION_SUMMARY_PATH = prepare_dataset(
    dataset_root=DATASET_ROOT, output_dir=PREPARED_DIR,
    max_development_frames=int(CONFIG['max_development_frames']),
    external_frames=int(CONFIG['external_frames']),
    external_view=CONFIG['external_view'],
    label_frames=int(CONFIG['object_label_frames']), seed=int(CONFIG['seed']),
)
print('Manifiesto:', MANIFEST_PATH)
print(PREPARATION_SUMMARY_PATH.read_text(encoding='utf-8'))

FileNotFoundError: Edita DATASET_ROOT para apuntar a PKLot: RUTA\AL\PKLot\EXTRAIDO

## 3. Revisar los datos y anotar objetos

Comprueba los conteos por split, vista y clima. Revisa todas las imágenes indicadas en `object_labeled_images.txt`, incluidas las que no tengan objetos. Añade una fila por caja a `object_boxes.csv` con clase `person`, `car` o `motorcycle` y coordenadas de píxel. Solo se anotan imágenes de test.

In [ ]:
records = read_manifest(MANIFEST_PATH)
counts = {}
for record in records:
    key = (record.split, record.view, record.weather)
    counts[key] = counts.get(key, 0) + 1
for key, count in sorted(counts.items()):
    print(f'{key[0]:13s} | {key[1]:7s} | {key[2]:7s} | {count:4d}')
ANNOTATIONS_DIR = PREPARED_DIR / 'annotations'
print('Imágenes para revisar:', ANNOTATIONS_DIR / 'object_labeled_images.txt')
print('Cajas manuales:', ANNOTATIONS_DIR / 'object_boxes.csv')

## 4. Homografía opcional

Para una vista superior, selecciona la vista y ejecuta la primera celda para ver una imagen de train. Después introduce cuatro puntos de una zona plana en orden: arriba-izquierda, arriba-derecha, abajo-derecha, abajo-izquierda. Las coordenadas son píxeles de esa imagen. La homografía es auxiliar y no condiciona la clasificación.

In [ ]:
VIEW_TO_CALIBRATE = None  # Ejemplo: 'UFPR04'; None omite esta etapa.
if VIEW_TO_CALIBRATE is not None:
    calibration_record = next(
        (r for r in records if r.view == VIEW_TO_CALIBRATE and r.split == 'train'), None
    )
    if calibration_record is None:
        raise ValueError(f'No hay imagen train para {VIEW_TO_CALIBRATE}.')
    calibration_image = cv2.imread(str(calibration_record.image_path), cv2.IMREAD_COLOR)
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(calibration_image, cv2.COLOR_BGR2RGB))
    plt.title(f'{VIEW_TO_CALIBRATE}: anota visualmente las cuatro esquinas')
    plt.axis('on')
    plt.show()
else:
    print('Homografía omitida; configura VIEW_TO_CALIBRATE para mostrar una imagen.')

In [ ]:
SOURCE_POINTS = []  # Cuatro pares [x, y], si se activa la calibración.
if VIEW_TO_CALIBRATE is not None:
    if len(SOURCE_POINTS) != 4:
        raise ValueError('SOURCE_POINTS debe contener cuatro pares [x, y].')
    CONFIG.setdefault('homography_source_points_by_view', {})[VIEW_TO_CALIBRATE] = SOURCE_POINTS
    CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'Puntos guardados para {VIEW_TO_CALIBRATE}.')
else:
    print('Homografía omitida; configura VIEW_TO_CALIBRATE y SOURCE_POINTS para usarla.')

## 5. Ejecutar los métodos

Se calculan Otsu y HOG+SVM; la ejecución completa añade embeddings DINOv2-S/14 y YOLOv8n. Los umbrales se ajustan en validación y test queda reservado para el informe final.

In [ ]:
RUN_YOLO = True
RUN_DINO = True
DEVICE = 'auto'  # También: 'cuda:0' o 'cpu'
RUN_DIR = OUTPUT_DIR / 'run'

# Deja ambos vacíos si aún no has terminado las anotaciones manuales.
BOXES_CSV = ''
LABELED_IMAGES_FILE = ''
CONFIG['object_annotations_csv'] = resolve_project_path(BOXES_CSV) if BOXES_CSV else ''
CONFIG['object_labeled_images_file'] = resolve_project_path(LABELED_IMAGES_FILE) if LABELED_IMAGES_FILE else ''
run_experiment(
    manifest_path=MANIFEST_PATH, output_dir=RUN_DIR, config=CONFIG,
    requested_device=DEVICE, with_yolo=RUN_YOLO, with_dino=RUN_DINO,
)
print('Resultados en:', RUN_DIR)

## 6. Consultar métricas y visualizar

Las métricas incluyen precisión, recall, F1, accuracy y TP/TN/FP/FN, separadas por split, vista y clima. Interpreta la evaluación de cajas según el soporte de cada clase: PKLot no aporta esas cajas y hay que anotarlas manualmente.

In [ ]:
metrics_path = RUN_DIR / 'occupancy_metrics.csv'
with metrics_path.open(newline='', encoding='utf-8-sig') as stream:
    for row in csv.DictReader(stream):
        print(row)
print((RUN_DIR / 'run_summary.json').read_text(encoding='utf-8'))
object_metrics_path = RUN_DIR / 'object_detection_metrics.csv'
if object_metrics_path.exists():
    with object_metrics_path.open(newline='', encoding='utf-8-sig') as stream:
        for row in csv.DictReader(stream):
            print('Objetos:', row)

overlay_files = sorted((RUN_DIR / 'overlays').glob('*.jpg'))
if overlay_files:
    image = cv2.imread(str(overlay_files[0]), cv2.IMREAD_COLOR)
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(overlay_files[0].name)
    plt.axis('off')
    plt.show()
else:
    print('No hay overlays disponibles.')

## Salidas

En `outputs/run/` se guardan métricas, predicciones por plaza, detecciones, overlays, el informe geométrico, configuración y versiones, y los embeddings/modelos lineales. El objetivo es comparar técnicas, explicar errores y documentar límites; no alcanzar una cifra de precisión predeterminada.